## LLM Settings

In [ ]:
import openai
from openai import OpenAI

# Modify related methods in the LLMEnhancer class

class LLMEnhancer:
    def __init__(self, config):
        self.config = config
        self.client = None
        self._initialize_client()

    def _initialize_client(self):
        """Initialize OpenAI client"""
        api_key = self.config['llm']['api_key']
        if api_key is None:
            print("Warning: OpenAI API key not set. LLM features will be disabled.")
            return

        try:
            self.client = OpenAI(api_key=api_key)
            print("OpenAI client initialized successfully")
        except Exception as e:
            print(f"Failed to initialize OpenAI client: {str(e)}")

    def generate_topic_name(self, keywords, dataset_info="", topic_id=None, topic_size=None, max_retries=3):
        """Generate meaningful names for topics"""
        if self.client is None:
            return f"主題: {', '.join(keywords[:3])}"

        # Prepare keyword string (including weights)
        if isinstance(keywords[0], tuple):
            # If in (word, weight) format
            keywords_str = ", ".join([f"{word}({weight:.4f})" for word, weight in keywords[:8]])
        else:
            # If just a list of words
            keywords_str = ", ".join(keywords[:8])

        # Construct prompt
        prompt = f"""你是一位資深的傳說對決遊戲分析師和社群輿情專家，同時具備BERTopic主題建模的技術知識。

【BERTopic技術背景】
你正在為BERTopic主題模型生成主題名稱。BERTopic通過以下步驟工作：
1. 文檔嵌入向量化
2. UMAP降維聚類
3. HDBSCAN主題群組識別
4. c-TF-IDF計算詞彙重要性

c-TF-IDF權重原理：
- 權重越高 = 該詞越能代表此主題
- 關鍵詞按權重排序，前面的詞更重要
- 權重反映詞彙在該主題的獨特性

【遊戲背景】
傳說對決：5v5 MOBA，時間範圍2024年7-12月
- 貨幣：金幣(買英雄)、點券(買造型/抽獎)
- 地圖：河蟹、野怪、防禦塔
- 商業：直購(直接購買) vs 抽獎機制
- IP聯動：如咒術迴戰五條悟造型
- 主播：黃帝、黑貓無敵、海牛、蘿蔔等知名實況主
- 英雄：牛牛(牛魔王)、夜叉等
**未提及內容請依專業知識補充**

【Few-shot Examples - 學習這些優質命名】

示例1：
關鍵詞：直購(0.8881), 購(0.8342), 送(0.8152), 希望直購(0.8139), 直購造型(0.7943)
分析步驟：
步驟1：前3高權重詞都指向"直購"概念
步驟2：玩家在討論購買方式，希望避免抽獎
步驟3：情緒是期待和訴求
步驟4：這是商業模式相關的用戶需求
步驟5：核心是對直購機制的強烈需求
命名：造型直購訴求

示例2：
關鍵詞：五條悟(0.6822), 想要條(0.5708), 抽到條(0.4763), 抽條(0.4751), 條黃帝(0.4517)
分析步驟：
步驟1：明確指向五條悟IP聯動造型
步驟2：玩家在討論獲得五條悟造型的願望
步驟3：情緒是渴望和期待
步驟4：這是IP聯動造型的需求表達
步驟5：核心是對特定IP角色的強烈喜愛
命名：五條悟造型渴望

示例3：
關鍵詞：難看(0.6446), 可惜(0.6422), 造型唯一不滿(0.6306), 醜(0.6159), 爛(0.5916)
分析步驟：
步驟1：全是負面評價詞彙
步驟2：玩家在批評造型的視覺效果
步驟3：情緒是失望和不滿
步驟4：這是對產品品質的負面反饋
步驟5：核心是造型外觀設計的批評
命名：造型外觀不滿

【分析任務框架】"""

        # Add only if topic_id and topic_size are present
        if topic_id is not None and topic_size is not None:
            prompt += f"""
主題ID：{topic_id}
主題規模：{topic_size}個文檔"""

        prompt += f"""
資料集類型：{dataset_info}
關鍵詞（按c-TF-IDF權重排序）：{keywords_str}

【Step-by-Step分析流程】
請按以下5個步驟進行思考，展示你的分析過程：

步驟1：權重分析
- 前5個最高權重的關鍵詞是什麼？
- 這些詞的權重差異說明什麼？

步驟2：語意歸納
- 這些高權重詞指向什麼核心概念？
- 是遊戲機制、商業行為、還是情緒表達？

步驟3：情緒識別
- 從關鍵詞中能感受到什麼情緒？
- 是正面(讚美、期待)還是負面(抱怨、失望)？

步驟4：行為判斷
- 玩家在這個主題中主要在做什麼？
- 是討論、抱怨、讚美、還是提出需求？

步驟5：連貫性檢查
- 前幾個高權重詞是否連貫？
- 如果不連貫，能找到共同的情緒或行為模式嗎？

【Chain of Thought輸出格式】
請用以下格式輸出你的思考過程：

分析過程：
步驟1：[權重分析結果]
步驟2：[語意歸納結果]
步驟3：[情緒識別結果]
步驟4：[行為判斷結果]
步驟5：[連貫性檢查結果]

核心概念：[一句話總結這個主題的核心]

主題名稱：[最終命名結果]

【重要】最後一行必須是"主題名稱：XXX"的格式，XXX是你的最終命名。

【命名要求】
- 遊戲元素優先：英雄名、地圖物件、裝備、造型名稱等具體元素最重要
- 避免形容詞濫用**：少用"強烈"、"高度"、"深度"等修飾詞
- 重點關注前3-5個高權重關鍵詞
- 重點關注遊戲術語和玩家語言
- 反映真實玩家行為和情緒強度
- 禁止"主題:"等前綴和特殊標點符號
- 如果關鍵詞真的不連貫，誠實命名為"綜合討論"類型

【優化原則】
- 控制名稱長度：建議4-8個中文字，最多不超過10字
- 主題要清晰且語意明確



- 主題名稱要簡潔且具體，避免包含分析內容"""

        for attempt in range(max_retries):
            try:
                response = self.client.chat.completions.create(
                    model=self.config['llm']['model'],
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=1000,  # Increase token count
                    temperature=0.4   # Slightly increase creativity
                )

                full_response = response.choices[0].message.content.strip()

                # Extract the final topic name
                if "主題名稱：" in full_response:
                    topic_name = full_response.split("主題名稱：")[-1].strip()
                elif "主題名稱:" in full_response:
                    topic_name = full_response.split("主題名稱:")[-1].strip()
                else:
                    # If output is not formatted, take the last line
                    topic_name = full_response.split('\n')[-1].strip()

                # Clean up potential prefixes
                topic_name = topic_name.replace("主題:", "").replace("Topic:", "").strip()

                if len(topic_name) > 0 and len(topic_name) <= 50:  # Relax length restrictions
                    return topic_name

            except Exception as e:
                print(f"Attempt {attempt + 1} failed: {str(e)}")
                if attempt == max_retries - 1:
                    print("All attempts failed, using fallback name")

        return f"主題: {keywords[0]}-{keywords[1] if len(keywords) > 1 else '相關'}"

    def enhance_model_topics(self, model, dataset_name=""):
        """Generate names for all topics in the model"""
        if self.client is None:
            print("LLM client not available, skipping topic enhancement")
            return {}

        dataset_info = self.config['data']['datasets'].get(dataset_name, {}).get('description', dataset_name)
        print(f"Enhancing topics for {dataset_info}...")

        topic_info = model.get_topic_info()
        non_outlier_topics = topic_info[topic_info['Topic'] != -1]['Topic'].tolist()

        enhanced_topics = {}

        for topic_id in non_outlier_topics:
            topic_keywords = model.get_topic(topic_id)
            keywords = [word for word, _ in topic_keywords[:8]]

            # Get topic size
            topic_size = topic_info[topic_info['Topic'] == topic_id]['Count'].values[0]

            # Use new function signature to pass more information
            topic_name = self.generate_topic_name(
                topic_keywords[:8],  # Pass (word, weight) tuples
                dataset_info,
                topic_id=topic_id,
                topic_size=topic_size
            )

            enhanced_topics[topic_id] = {
                'name': topic_name,
                'keywords': keywords,
                'original_keywords': [f"{word}({weight:.3f})" for word, weight in topic_keywords[:5]]
            }

            print(f"Topic {topic_id}: {topic_name}")
            time.sleep(0.5)  # Avoid API limits

        return enhanced_topics

# Initialize LLM Enhancer
enhancer = LLMEnhancer(PIPELINE_CONFIG)
print("LLMEnhancer initialized")
print("功能：根據主題關鍵詞自動生成適合商業報告的中文主題名稱")

OpenAI client initialized successfully
LLMEnhancer initialized
功能：根據主題關鍵詞自動生成適合商業報告的中文主題名稱
